In [1]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from collections import deque
import sys
import time


In [2]:
# Cell 2: HybridPowerFlowOptimizer Class (Net Injection Model with Slack Bus)
class HybridPowerFlowOptimizer:
    """
    Optimizes power flow using a hybrid metaheuristic approach, handling
    buses with fixed load and controllable generation based on user specification.
    Uses a SLACK BUS (must be a generator) to enforce power balance.
    Allows zero-cost load shedding ONLY for buses designated as pure loads.
    """

    # MODIFIED __init__ method: Add slack_bus_index parameter and related variables
    def __init__(self, A_matrix, line_limits, gen_costs_full, gen_limits_min_full, gen_limits_max_full,
                 initial_B_net, fixed_load_full,
                 gen_indices, load_indices,
                 slack_bus_index, # <-- NEW: 0-based index of the designated slack bus
                 memory_size=100):
        """
        Initializes the optimizer.

        Args:
            A_matrix (np.ndarray): System matrix (e.g., PTDF). Shape (num_lines, num_buses).
            line_limits (np.ndarray): Absolute power flow limits for each line. Shape (num_lines,).
            gen_costs_full (np.ndarray): Cost coefficients ($/MW change) for ALL buses. Shape (num_buses,).
            gen_limits_min_full (np.ndarray): Min generation (Pg) limit for ALL buses. Shape (num_buses,).
            gen_limits_max_full (np.ndarray): Max generation (Pg) limit for ALL buses. Shape (num_buses,).
            initial_B_net (np.ndarray): Initial NET bus injections (Pg - Pl). Shape (num_buses, 1) or (num_buses,).
            fixed_load_full (np.ndarray): Fixed load demand (Pl >= 0) for ALL buses. Shape (num_buses, 1) or (num_buses,).
            gen_indices (list or np.ndarray): 0-based indices for generator buses.
            load_indices (list or np.ndarray): 0-based indices for load-only buses.
            slack_bus_index (int): 0-based index of the designated slack bus (must be in gen_indices).
            memory_size (int): Size of the memory for storing past feasible solutions.
        """
        self.A = A_matrix
        self.line_limits = np.array(line_limits, dtype=np.float64)
        self.num_lines = A_matrix.shape[0]
        self.num_buses = A_matrix.shape[1]
        self.tolerance = 1e-6 # Numerical tolerance for checks

        # --- Store Initial Net Injection, Fixed Load, and Indices ---
        self.initial_B_net = initial_B_net.copy().flatten() # Ensure it's a flat array
        self.fixed_load = fixed_load_full.copy().flatten() # Ensure it's a flat array
        # Ensure fixed loads are non-negative
        if np.any(self.fixed_load < 0):
            print("Warning: Fixed loads should be non-negative. Clamping negative loads to zero.")
            self.fixed_load = np.maximum(0, self.fixed_load)

        self.gen_indices = np.array(sorted(gen_indices), dtype=int)
        self.load_indices = np.array(sorted(load_indices), dtype=int)

        # --- Slack Bus Validation ---
        self.slack_bus_index = int(slack_bus_index)
        if len(self.gen_indices) == 0:
             raise ValueError("Cannot assign a slack bus when no generators are specified.")
        if self.slack_bus_index not in self.gen_indices:
            raise ValueError(f"Specified Slack Bus index ({self.slack_bus_index}) is not in the list of Generator indices ({self.gen_indices}).")
        print(f"Using Bus {self.slack_bus_index+1} as the Slack Bus.")

        # Identify non-slack generators (used for applying constraints separately)
        self.non_slack_gen_indices = np.setdiff1d(self.gen_indices, [self.slack_bus_index], assume_unique=True)
        # --- End Slack Bus Validation ---


        # Validation (Generator and Load Indices must cover all buses without overlap)
        all_indices = np.concatenate((self.gen_indices, self.load_indices))
        if len(np.unique(all_indices)) != self.num_buses or len(all_indices) != self.num_buses:
             raise ValueError("Generator and Load indices do not form a complete, non-overlapping set of all buses.")

        print(f"Using specified {len(self.gen_indices)} generator buses (indices: {self.gen_indices})")
        print(f"Using specified {len(self.load_indices)} load-only buses (indices: {self.load_indices})")

        # --- Store Generator-Specific Data (Pg limits and costs) ---
        # These arrays will correspond to the order in self.gen_indices
        num_gens = len(self.gen_indices)
        self.gen_costs_only = np.zeros(num_gens)
        self.gen_limits_Pg_min_only = np.zeros(num_gens) # Store Pg limits
        self.gen_limits_Pg_max_only = np.zeros(num_gens) # Store Pg limits

        # Find the *local* index of the slack bus within the gen_indices array
        # This is needed to easily access its limits later from the _only arrays
        self.slack_bus_local_gen_idx_ = np.where(self.gen_indices == self.slack_bus_index)[0]
        if len(self.slack_bus_local_gen_idx_) == 0:
             # This should theoretically not happen due to validation above, but safety check
             raise ValueError("Internal Error: Could not find slack bus local index.")
        self.slack_bus_local_gen_idx = self.slack_bus_local_gen_idx_[0]


        if num_gens > 0:
            # Ensure input arrays have the correct full size (num_buses)
            flat_gc = np.array(gen_costs_full).flatten()
            flat_gmin = np.array(gen_limits_min_full).flatten()
            flat_gmax = np.array(gen_limits_max_full).flatten()
            if len(flat_gc)!=self.num_buses or len(flat_gmin)!=self.num_buses or len(flat_gmax)!=self.num_buses:
                raise ValueError(f"Generator cost/limit array length mismatch (Expected {self.num_buses})")

            # Extract data only for the specified generator indices
            self.gen_costs_only = flat_gc[self.gen_indices]
            self.gen_limits_Pg_min_only = flat_gmin[self.gen_indices]
            self.gen_limits_Pg_max_only = flat_gmax[self.gen_indices]

            # Validate Pg limits
            if np.any(self.gen_limits_Pg_max_only < self.gen_limits_Pg_min_only):
                raise ValueError("Generator Max Pg limit cannot be less than Min Pg limit.")

        # --- Calculate and Store NET Injection Limits (Bnet = Pg - Pl) ---
        # These limits also correspond to the order in self.gen_indices
        # For Generators: B_net = Pg - Pl_fixed => Pg_min - Pl <= B_net <= Pg_max - Pl
        self.gen_limits_Bnet_min_only = np.zeros(num_gens)
        self.gen_limits_Bnet_max_only = np.zeros(num_gens)
        if num_gens > 0:
            fixed_load_at_gens = self.fixed_load[self.gen_indices]
            self.gen_limits_Bnet_min_only = self.gen_limits_Pg_min_only - fixed_load_at_gens
            self.gen_limits_Bnet_max_only = self.gen_limits_Pg_max_only - fixed_load_at_gens

        # For Load-Only buses: B_net = 0 - Pl => -Pl_fixed <= B_net <= 0 (Shedding allowed)
        # These limits correspond to the order in self.load_indices
        num_loads = len(self.load_indices)
        self.load_limits_Bnet_min_only = np.zeros(num_loads) # Min net injection = -Pl_fixed
        self.load_limits_Bnet_max_only = np.zeros(num_loads) # Max net injection = 0 (fully shed)
        if num_loads > 0:
            fixed_load_at_loads = self.fixed_load[self.load_indices]
            self.load_limits_Bnet_min_only = 0.0 - fixed_load_at_loads # B can go down to -Pl

        # --- Adjust Initial NET Injection State if Necessary ---
        # Ensures the starting point for the optimization respects limits as much as possible
        # and is balanced using the slack bus.
        needs_adjust = False
        # 1. Clamp initial NET injection at ALL generator buses (including slack) if implied Pg is outside limits
        if num_gens > 0:
            initial_Bnet_at_gens = self.initial_B_net[self.gen_indices]
            # Clip B_net based on calculated net limits for generators
            clipped_Bnet_gens = np.clip(initial_Bnet_at_gens,
                                        self.gen_limits_Bnet_min_only,
                                        self.gen_limits_Bnet_max_only)
            if np.any(np.abs(initial_Bnet_at_gens - clipped_Bnet_gens) > self.tolerance):
                print("Warning: Initial net injection at generator bus(es) implies Pg outside limits. Clamping net injection...")
                self.initial_B_net[self.gen_indices] = clipped_Bnet_gens
                needs_adjust = True

        # 2. Clamp initial NET injection at load buses (should be <= 0)
        if num_loads > 0:
            initial_Bnet_at_loads = self.initial_B_net[self.load_indices]
            # Clip B_net based on calculated net limits for loads (-Pl <= B <= 0)
            clipped_Bnet_loads = np.clip(initial_Bnet_at_loads,
                                          self.load_limits_Bnet_min_only,
                                          self.load_limits_Bnet_max_only)
            if np.any(np.abs(initial_Bnet_at_loads - clipped_Bnet_loads) > self.tolerance):
                print("Warning: Initial net injection at load bus(es) outside limits [-Pl, 0]. Clamping...")
                self.initial_B_net[self.load_indices] = clipped_Bnet_loads
                needs_adjust = True


        # 3. Balance initial NET injections (sum should be close to zero) by adjusting ONLY THE SLACK BUS
        required_total_injection = 0.0
        current_total_injection = np.sum(self.initial_B_net)
        difference_init = required_total_injection - current_total_injection

        if abs(difference_init) > self.tolerance * self.num_buses:
            print(f"Warning: Initial NET injections sum to {current_total_injection:.4f} (≠ 0). Adjusting SLACK BUS net injection to balance...")
            # Adjust slack bus B_net
            adjusted_Bnet_slack = self.initial_B_net[self.slack_bus_index] + difference_init
            # Clip adjustment based on *net injection* limits for the slack bus
            slack_min_Bnet = self.gen_limits_Bnet_min_only[self.slack_bus_local_gen_idx]
            slack_max_Bnet = self.gen_limits_Bnet_max_only[self.slack_bus_local_gen_idx]
            clipped_adjusted_Bnet_slack = np.clip(adjusted_Bnet_slack, slack_min_Bnet, slack_max_Bnet)

            # Calculate how much adjustment was actually applied after clipping
            actual_adjustment_applied = clipped_adjusted_Bnet_slack - self.initial_B_net[self.slack_bus_index]
            self.initial_B_net[self.slack_bus_index] = clipped_adjusted_Bnet_slack
            remaining_diff = difference_init - actual_adjustment_applied

            # Warn if clipping prevented full balance
            if abs(remaining_diff) > self.tolerance:
                print(f"Warning: Could not fully balance initial state due to SLACK BUS net injection limits. Remaining imbalance: {remaining_diff:.4f}")
            needs_adjust = True

        if needs_adjust:
            print("-> Adjusted initial Net Injection (B) state used for optimization:", np.round(self.initial_B_net, 4))
        # --- End Initial State Handling ---

        # Optimization parameters
        self.memory = deque(maxlen=memory_size)
        self.stagnation_threshold = 50 # Iterations without improvement before diversification
        self.diversification_fraction = 0.2 # Fraction of population to replace during diversification

    # --- Methods _apply_constraints, _calculate_fitness, _is_feasible, etc. ---
    # These now operate on NET INJECTION (B) but use the derived Bnet limits and slack bus logic

    # MODIFIED _apply_constraints method: Use Slack Bus for balancing
    def _apply_constraints(self, solution_vector):
        """
        Applies NET injection limits and enforces power balance using the SLACK BUS.
        Clips load buses and non-slack generators.
        Adjusts slack bus B_net to enforce sum(B_net) = 0, WITHOUT clipping slack here.
        """
        sol_Bnet = solution_vector.copy().flatten() # Work with a flat copy
        num_loads = len(self.load_indices)
        num_non_slack_gens = len(self.non_slack_gen_indices)

        # 1. Apply Load-Only Bus Net Injection Limits (Shedding: -Pl <= B <= 0)
        if num_loads > 0:
            # Use local indices corresponding to load_indices to access correct limits
            sol_Bnet[self.load_indices] = np.clip(sol_Bnet[self.load_indices],
                                                self.load_limits_Bnet_min_only, # Indices match load_indices
                                                self.load_limits_Bnet_max_only) # Indices match load_indices

        # 2. Apply NON-SLACK Generator Bus Net Injection Limits (Pg_min - Pl <= B <= Pg_max - Pl)
        if num_non_slack_gens > 0:
             # Need local indices for non-slack generators within the gen_limits arrays
             non_slack_gen_local_indices = np.where(np.isin(self.gen_indices, self.non_slack_gen_indices))[0]
             sol_Bnet[self.non_slack_gen_indices] = np.clip(sol_Bnet[self.non_slack_gen_indices],
                                                             self.gen_limits_Bnet_min_only[non_slack_gen_local_indices],
                                                             self.gen_limits_Bnet_max_only[non_slack_gen_local_indices])

        # 3. Enforce Power Balance (Adjust ONLY SLACK BUS Net Injection)
        # Balance requires sum of NET injections = 0
        required_total_Bnet = 0.0
        current_total_Bnet = np.sum(sol_Bnet)
        difference = required_total_Bnet - current_total_Bnet # Amount needed to balance

        # Apply the entire difference to the slack bus B_net
        # We specifically DO NOT clip the slack bus B_net here.
        # This mimics the behavior of a slack bus in traditional load flow.
        # Its limits will be checked/penalized in the fitness function.
        sol_Bnet[self.slack_bus_index] += difference

        # Return the adjusted B_net vector reshaped as a column vector
        return sol_Bnet.reshape(-1, 1)


    # MODIFIED _calculate_fitness method: Add Slack Bus Limit Penalty
    def _calculate_fitness(self, solution_Bnet):
        """
        Calculates fitness (cost/penalty) of a given net injection solution (B_net).
        Includes penalties for line limit violations, non-slack generator limit violations,
        slack bus limit violations, load bus limit violations, and power imbalance.
        Also includes weighted objective terms for minimizing generator deviation and cost.
        """
        sol_Bnet_flat = solution_Bnet.flatten() # Work with flat array

        # Basic check for invalid numbers
        if not np.all(np.isfinite(sol_Bnet_flat)):
            return np.inf # Return infinite fitness for invalid solutions

        # Calculate line flows: C = A * B_net
        try:
            C = np.dot(self.A, sol_Bnet_flat)
            flows = C.flatten()
            if not np.all(np.isfinite(C)): # Check if flows calculation resulted in NaN/Inf
                return np.inf
        except ValueError: # Handle potential dimension mismatch errors
            return np.inf

        # --- Penalty Components ---
        penalty_multiplier = 1e10 # Large multiplier for constraint violations

        # 1. Line Limit Penalty (Based on flows from B_net)
        line_violations = np.maximum(0, np.abs(flows) - (self.line_limits + self.tolerance))
        line_violation_penalty = penalty_multiplier * np.sum(line_violations**2) # Squared violation

        # 2. NON-SLACK Generator Net Injection Limit Penalty
        gen_limit_penalty = 0.0
        num_non_slack_gens = len(self.non_slack_gen_indices)
        if num_non_slack_gens > 0:
            # Get local indices within gen_limits arrays for non-slack generators
            non_slack_gen_local_indices = np.where(np.isin(self.gen_indices, self.non_slack_gen_indices))[0]
            non_slack_gen_Bnet_values = sol_Bnet_flat[self.non_slack_gen_indices]
            # Calculate violations below min and above max limits
            violations_lower_g = np.maximum(0, self.gen_limits_Bnet_min_only[non_slack_gen_local_indices] - non_slack_gen_Bnet_values + self.tolerance)
            violations_upper_g = np.maximum(0, non_slack_gen_Bnet_values - self.gen_limits_Bnet_max_only[non_slack_gen_local_indices] - self.tolerance)
            gen_limit_penalty = penalty_multiplier * (np.sum(violations_lower_g**2) + np.sum(violations_upper_g**2))

        # 3. SLACK BUS Net Injection Limit Penalty <--- NEW PENALTY TERM
        slack_limit_penalty = 0.0
        slack_Bnet_value = sol_Bnet_flat[self.slack_bus_index]
        # Get slack bus limits using its pre-calculated local index
        slack_min_Bnet = self.gen_limits_Bnet_min_only[self.slack_bus_local_gen_idx]
        slack_max_Bnet = self.gen_limits_Bnet_max_only[self.slack_bus_local_gen_idx]
        # Calculate violations below min and above max limits for the slack bus
        violation_lower_s = np.maximum(0, slack_min_Bnet - slack_Bnet_value + self.tolerance)
        violation_upper_s = np.maximum(0, slack_Bnet_value - slack_max_Bnet - self.tolerance)
        # Use square of violation for penalty
        slack_limit_penalty = penalty_multiplier * (violation_lower_s**2 + violation_upper_s**2)


        # 4. Load-Only Net Injection Limit Penalty (Unchanged logic)
        load_limit_penalty = 0.0
        num_loads = len(self.load_indices)
        if num_loads > 0:
            load_Bnet_values = sol_Bnet_flat[self.load_indices]
            # Calculate violations below min and above max limits
            violations_lower_l = np.maximum(0, self.load_limits_Bnet_min_only - load_Bnet_values + self.tolerance) # Limits match load_indices order
            violations_upper_l = np.maximum(0, load_Bnet_values - self.load_limits_Bnet_max_only - self.tolerance) # Limits match load_indices order
            load_limit_penalty = penalty_multiplier * (np.sum(violations_lower_l**2) + np.sum(violations_upper_l**2))

        # 5. Power Balance Penalty (Keep this, even if small, for robustness)
        # Although _apply_constraints forces balance via slack, numerical issues might remain.
        balance_violation = abs(np.sum(sol_Bnet_flat))
        # Only apply penalty if violation significantly exceeds tolerance
        balance_penalty = penalty_multiplier * (balance_violation**2) if balance_violation > self.tolerance * 10 else 0.0


        # --- Objective Components ---
        # These objectives are applied based on the change from the initial B_net state.
        # They are applied to ALL generators, including the slack bus.
        # (Could be modified to exclude slack if the objective was different, e.g., minimizing total fuel cost).

        # 6. Generator Deviation Penalty (Minimize change in B_net for ALL generators)
        generator_deviation = self._get_generator_deviation(solution_Bnet) # Uses B_net
        deviation_penalty_component = 1e5 * generator_deviation # Tunable weight (High weight = prioritize staying close to initial)

        # 7. Generator Rescheduling Cost (Minimize cost of change in B_net for ALL generators)
        gen_rescheduling_cost = self._get_rescheduling_cost(solution_Bnet) # Uses B_net
        gen_cost_weight = 0.1 # Tunable weight (Lower weight = cost is less important than deviation/constraints)

        # --- Total Fitness ---
        # Sum of all penalties and weighted objectives
        fitness = (line_violation_penalty + gen_limit_penalty + slack_limit_penalty +
                   load_limit_penalty + balance_penalty +
                   deviation_penalty_component + gen_cost_weight * gen_rescheduling_cost)

        # Ensure fitness is a finite value
        return fitness if np.isfinite(fitness) else np.inf

    # MODIFIED _is_feasible method: Check Slack Bus Limits
    def _is_feasible(self, solution_Bnet, verbose=False):
        """
        Checks if a NET injection solution meets all hard constraints (incl. SLACK limits).
        Hard constraints are: Line limits, non-slack generator limits, slack generator limits,
        load bus limits, and power balance (within tolerance).
        """
        if solution_Bnet is None:
            if verbose: print("DEBUG (_is_feasible): Input solution is None.")
            return False
        sol_Bnet_flat = solution_Bnet.flatten()
        if not np.all(np.isfinite(sol_Bnet_flat)):
            if verbose: print("DEBUG (_is_feasible): Solution contains non-finite values.")
            return False

        # Check Line Limits
        line_ok = False
        flows = np.array([])
        try:
            C = np.dot(self.A, sol_Bnet_flat); flows = C.flatten()
            if not np.all(np.isfinite(C)): raise ValueError("Flows NaN/Inf")
            # Check if absolute flow is within limit + tolerance
            line_ok = np.all(np.abs(flows) <= self.line_limits + self.tolerance)
        except Exception as e:
            line_ok = False;
            if verbose: print(f"DEBUG (_is_feasible): Line flow calculation error: {e}")


        # Check NON-SLACK Generator Net Injection Limits
        non_slack_gen_ok = True # Assume OK if no non-slack generators
        if len(self.non_slack_gen_indices) > 0:
            non_slack_gen_local_indices = np.where(np.isin(self.gen_indices, self.non_slack_gen_indices))[0]
            non_slack_gen_Bnet_values = sol_Bnet_flat[self.non_slack_gen_indices]
            non_slack_gen_ok = np.all(non_slack_gen_Bnet_values >= self.gen_limits_Bnet_min_only[non_slack_gen_local_indices] - self.tolerance) and \
                               np.all(non_slack_gen_Bnet_values <= self.gen_limits_Bnet_max_only[non_slack_gen_local_indices] + self.tolerance)

        # Check SLACK BUS Net Injection Limits <--- ADDED CHECK
        slack_bus_ok = True # Assume OK initially
        slack_Bnet_value = sol_Bnet_flat[self.slack_bus_index]
        slack_min_Bnet = self.gen_limits_Bnet_min_only[self.slack_bus_local_gen_idx]
        slack_max_Bnet = self.gen_limits_Bnet_max_only[self.slack_bus_local_gen_idx]
        slack_bus_ok = (slack_Bnet_value >= slack_min_Bnet - self.tolerance) and \
                       (slack_Bnet_value <= slack_max_Bnet + self.tolerance)

        # Check Load-Only Net Injection Limits
        load_ok = True # Assume OK if no load-only buses
        if len(self.load_indices) > 0:
            load_Bnet_values = sol_Bnet_flat[self.load_indices]
            load_ok = np.all(load_Bnet_values >= self.load_limits_Bnet_min_only - self.tolerance) and \
                      np.all(load_Bnet_values <= self.load_limits_Bnet_max_only + self.tolerance)

        # Check Power Balance (Sum of Net Injections should be close to zero)
        bal_ok = abs(np.sum(sol_Bnet_flat)) < self.tolerance * self.num_buses # Allow slightly larger tolerance

        # Overall feasibility: all checks must pass
        feasible = line_ok and non_slack_gen_ok and slack_bus_ok and load_ok and bal_ok

        # Optional verbose output for debugging
        if verbose or not feasible:
            print(f"--- Feasibility Check {'FAILED' if not feasible else 'PASSED'} (Slack Bus Style) ---")
            if not line_ok: print(f"  Line constraints failed.")
            else: print("  Line constraints met.")
            if not non_slack_gen_ok: print(f"  NON-SLACK Generator net injection limits failed (Buses: {self.non_slack_gen_indices+1}).")
            else: print("  NON-SLACK Generator net injection limits met.")
            if not slack_bus_ok: print(f"  SLACK BUS ({self.slack_bus_index+1}) net injection limits failed (Value={slack_Bnet_value:.4f}, Limits=[{slack_min_Bnet:.4f}, {slack_max_Bnet:.4f}]).") # More detail
            else: print(f"  SLACK BUS ({self.slack_bus_index+1}) net injection limits met.")
            if not load_ok: print(f"  Load-only net injection limits failed (Buses: {self.load_indices+1}).")
            else: print("  Load-only net injection limits met.")
            if not bal_ok: print(f"  Power balance check failed, sum(B_net)={np.sum(sol_Bnet_flat):.8f}")
            else: print("  Power balance met.")
            print("-" * 40)
        return feasible

    # --- Helper methods for objectives (operate on B_net) ---
    # These calculate parts of the fitness function based on deviation from initial state.
    # They operate on ALL generators by default.
    def _get_generator_deviation(self, solution_Bnet):
        """Calculates sum of absolute changes in B_net for ALL generators."""
        sol_Bnet_flat = solution_Bnet.flatten()
        if len(self.gen_indices) == 0: return 0.0
        try:
            # Compare current B_net with initial B_net for ALL generator indices
            if np.max(self.gen_indices) >= len(sol_Bnet_flat) or np.max(self.gen_indices) >= len(self.initial_B_net):
                 print("Warning (_get_generator_deviation): Index mismatch.")
                 return np.inf # Index out of bounds
            # Calculate absolute difference for each generator and sum them up
            return np.sum(np.abs(sol_Bnet_flat[self.gen_indices] - self.initial_B_net[self.gen_indices]))
        except IndexError:
            print("Warning (_get_generator_deviation): IndexError.")
            return np.inf

    def _get_rescheduling_cost(self, solution_Bnet):
        """Calculates cost based on changes in B_net for ALL generators."""
        sol_Bnet_flat = solution_Bnet.flatten()
        if len(self.gen_indices) == 0: return 0.0
        try:
            # Use gen_costs_only (cost per MW change in Pg, which equals change in B_net)
            if len(self.gen_costs_only) != len(self.gen_indices) or \
               np.max(self.gen_indices) >= len(sol_Bnet_flat) or \
               np.max(self.gen_indices) >= len(self.initial_B_net):
                print("Warning (_get_rescheduling_cost): Index/length mismatch.")
                return np.inf # Index or length mismatch

            # Deviation is absolute change in B_net for each generator
            deviation_Bnet = np.abs(sol_Bnet_flat[self.gen_indices] - self.initial_B_net[self.gen_indices])
            # Cost is sum of (cost_per_MW * deviation_in_MW) for each generator
            cost = np.sum(self.gen_costs_only * deviation_Bnet)
            return cost
        except IndexError:
            print("Warning (_get_rescheduling_cost): IndexError.")
            return np.inf

    # --- Random Solution Generation (operates on B_net) ---
    def _generate_random_solution(self):
        """
        Generates a new random NET injection solution, respecting individual limits
        before applying the balancing constraint.
        """
        # Start with the (potentially adjusted) initial B_net state
        rand_sol_Bnet = self.initial_B_net.copy()
        n_gens = len(self.gen_indices)
        n_loads = len(self.load_indices)

        # Perturb Generator Net Injection within their Bnet limits
        if n_gens > 0:
            # Calculate the operational range for each generator's B_net
            gen_Bnet_range = np.maximum(self.tolerance, self.gen_limits_Bnet_max_only - self.gen_limits_Bnet_min_only)
            perturbation_factor = 0.2 # How much to perturb relative to the range
            # Generate random perturbations (positive or negative)
            perturbations_g = (np.random.rand(n_gens) - 0.5) * 2 * gen_Bnet_range * perturbation_factor
            # Apply perturbations and clip to ensure they stay within individual B_net limits
            rand_sol_Bnet[self.gen_indices] = np.clip(self.initial_B_net[self.gen_indices] + perturbations_g,
                                                      self.gen_limits_Bnet_min_only,
                                                      self.gen_limits_Bnet_max_only)

        # Perturb Load-Only Net Injection (Allowing shedding towards 0)
        if n_loads > 0:
            # Range is from min Bnet (-Pl) up to max Bnet (0)
            load_Bnet_range = self.load_limits_Bnet_max_only - self.load_limits_Bnet_min_only # Should be Pl (>=0)
            shedding_factor = 0.1 # Max % of range to perturb by (towards 0)
            # Perturbation increases Bnet (reduces load magnitude, i.e., sheds load)
            perturbations_l = np.random.rand(n_loads) * load_Bnet_range * shedding_factor
            # Apply perturbations and clip to ensure B_net stays within [-Pl, 0]
            rand_sol_Bnet[self.load_indices] = np.clip(self.initial_B_net[self.load_indices] + perturbations_l,
                                                       self.load_limits_Bnet_min_only,
                                                       self.load_limits_Bnet_max_only)

        # Apply constraints (including slack bus balancing) to the randomly perturbed solution
        return self._apply_constraints(rand_sol_Bnet)

    # --- Local Search (operates on B_net) ---
    def _apply_local_search(self, solution_Bnet, iteration, max_iterations):
        """
        Applies a simple local search heuristic to refine a NET injection solution.
        Randomly perturbs one bus's B_net and accepts if fitness improves.
        """
        current_solution_Bnet = solution_Bnet.copy()
        current_fitness = self._calculate_fitness(current_solution_Bnet)
        # Don't try local search on infeasible solutions
        if not np.isfinite(current_fitness):
            return solution_Bnet

        # Decrease step size as optimization progresses
        progress_ratio = iteration / max_iterations
        step_scale_factor = 0.1 * (1.0 - progress_ratio) + 0.01 * progress_ratio # Adaptive step size
        num_attempts = min(5, int(np.sqrt(self.num_buses))) # Number of perturbation attempts

        for _ in range(num_attempts):
            # Choose a random bus to perturb
            idx_to_perturb = random.randrange(self.num_buses)
            perturbed_solution_Bnet = current_solution_Bnet.copy()
            perturb_range = 1.0 # Default range

            # Determine perturbation range based on Bnet limits of the chosen bus
            if idx_to_perturb in self.gen_indices:
                # Find local index for this generator
                gen_idx_local = np.where(self.gen_indices == idx_to_perturb)[0][0]
                op_range = self.gen_limits_Bnet_max_only[gen_idx_local] - self.gen_limits_Bnet_min_only[gen_idx_local]
                perturb_range = max(self.tolerance, op_range) * step_scale_factor
            elif idx_to_perturb in self.load_indices:
                 # Find local index for this load
                load_idx_local = np.where(self.load_indices == idx_to_perturb)[0][0]
                op_range = self.load_limits_Bnet_max_only[load_idx_local] - self.load_limits_Bnet_min_only[load_idx_local]
                perturb_range = max(self.tolerance, op_range) * step_scale_factor

            # Generate perturbation (positive or negative)
            perturbation = (random.random() - 0.5) * 2 * perturb_range
            # Apply perturbation to the chosen bus's B_net
            perturbed_solution_Bnet[idx_to_perturb, 0] += perturbation

            # Apply constraints (including slack balancing) to the perturbed solution
            refined_solution_Bnet = self._apply_constraints(perturbed_solution_Bnet)
            # Calculate fitness of the refined solution
            refined_fitness = self._calculate_fitness(refined_solution_Bnet)

            # If the refined solution is better, accept it
            if np.isfinite(refined_fitness) and refined_fitness < current_fitness:
                current_solution_Bnet = refined_solution_Bnet
                current_fitness = refined_fitness

        return current_solution_Bnet

    # --- Optimization Loop (operates on B_net) ---
    def optimize(self, B_unused, iterations=200, population_size=30):
        """
        Main optimization loop using hybrid metaheuristics (operates on B_net).
        The B_unused argument is kept for potential compatibility but is not used here.
        """
        print(f"Starting Optimization: Pop={population_size}, Iterations={iterations}")
        # Start with the adjusted initial B_net from __init__
        initial_state_constrained = self._apply_constraints(self.initial_B_net)
        # Generate initial population, including the constrained initial state
        initial_population = [self._generate_random_solution() for _ in range(population_size)]
        initial_population[0] = initial_state_constrained # Ensure initial state is included

        # Evaluate fitness of the initial population
        fitness_values = [self._calculate_fitness(sol) for sol in initial_population]

        # Find the best initial solution
        best_idx = np.argmin(fitness_values) if np.any(np.isfinite(fitness_values)) else -1
        if best_idx != -1:
            best_solution_Bnet = initial_population[best_idx].copy()
            best_fitness = fitness_values[best_idx]
        else: # Handle case where initial population is entirely invalid
            print("ERROR: No finite fitness solutions found in initial population.")
            # Return the initial state (even if infeasible) and empty history
            try: C_initial = np.dot(self.A, self.initial_B_net)
            except Exception: C_initial = np.full((self.num_lines, 1), np.nan)
            return self.initial_B_net.reshape(-1, 1), [], C_initial

        print(f"Initial Best Fitness: {best_fitness:.4e}")
        fitness_history = [best_fitness] # Track best fitness over iterations

        # --- Adaptive Hybridization Setup ---
        algorithm_names = ['OOA', 'KHA', 'SHO'] # Names of the metaheuristics used
        # Track success and attempts for each algorithm
        algo_success_count = {name: 0 for name in algorithm_names}
        algo_attempts_count = {name: 0 for name in algorithm_names}
        # Dynamically adjust probabilities based on success rates
        algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}
        adaptation_rate = 0.05 # How quickly probabilities adapt

        # --- Stagnation Tracking ---
        stagnation_counter = 0
        last_best_fitness = best_fitness

        # --- Main Optimization Loop ---
        for iteration in range(iterations):
            new_population = [] # Store solutions for the next generation
            current_fitness_values = [] # Store fitness values for the new population

            # --- Adaptive Hybridization Logic ---
            # Calculate success rates and update algorithm selection probabilities
            total_attempts = sum(algo_attempts_count.values())
            if total_attempts > 0:
                success_rates = {name: algo_success_count[name] / algo_attempts_count[name] if algo_attempts_count[name] > 0 else 0 for name in algorithm_names}
                total_rate = sum(success_rates.values())
                if total_rate > 1e-6: # Avoid division by zero if no successes yet
                    # Target probabilities based on current success rates
                    target_probabilities = {name: rate / total_rate for name, rate in success_rates.items()}
                    # Update probabilities using adaptation rate
                    for name in algorithm_names:
                        algo_probabilities[name] = (1 - adaptation_rate) * algo_probabilities[name] + adaptation_rate * target_probabilities[name]
                    # Normalize probabilities to sum to 1
                    prob_sum = sum(algo_probabilities.values())
                    if prob_sum > 1e-6:
                        algo_probabilities = {name: p / prob_sum for name, p in algo_probabilities.items()}
                    else: # Reset if sum is too small
                        algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}
                # Use updated probabilities as weights for choosing algorithms
                current_weights = [algo_probabilities[name] for name in algorithm_names]
            else: # Use equal weights initially
                current_weights = [1.0/len(algorithm_names)] * len(algorithm_names)
            # --- End Adaptive Logic ---

            # --- Generate New Population ---
            for i in range(population_size):
                # Select an algorithm based on current probabilities
                try:
                    algorithm = random.choices(algorithm_names, weights=current_weights, k=1)[0]
                except ValueError: # Handle potential numerical issues with weights
                    algorithm = random.choice(algorithm_names)
                algo_attempts_count[algorithm] += 1 # Track attempt

                current_solution_Bnet = initial_population[i] # Get parent solution

                # Apply selected global search metaheuristic
                if algorithm == 'OOA':
                    candidate_Bnet = self._apply_orcas_optimization(initial_population, i, best_solution_Bnet, iteration, iterations)
                elif algorithm == 'KHA':
                    candidate_Bnet = self._apply_krill_herd(initial_population, i, best_solution_Bnet, iteration, iterations)
                else: # SHO
                    candidate_Bnet = self._apply_spotted_hyena(initial_population, i, best_solution_Bnet, iteration, iterations)

                # Apply local search to refine the candidate solution
                refined_candidate_Bnet = self._apply_local_search(candidate_Bnet, iteration, iterations)

                # Apply constraints (including slack balancing) to the refined solution
                new_solution_Bnet = self._apply_constraints(refined_candidate_Bnet)
                new_population.append(new_solution_Bnet) # Add to next generation

                # Evaluate fitness of the new solution
                new_fitness = self._calculate_fitness(new_solution_Bnet)
                current_fitness_values.append(new_fitness)

                # Update overall best solution found so far
                if np.isfinite(new_fitness) and new_fitness < best_fitness:
                    best_fitness = new_fitness
                    best_solution_Bnet = new_solution_Bnet.copy()
                    algo_success_count[algorithm] += 1 # Track success
            # --- End Population Generation ---

            # Replace old population with the new one
            initial_population = new_population
            fitness_values = current_fitness_values
            # Record the best fitness for this iteration
            if np.isfinite(best_fitness):
                fitness_history.append(best_fitness)

            # --- Stagnation & Diversification Logic ---
            # Check if the best fitness has improved significantly
            if abs(best_fitness - last_best_fitness) < self.tolerance * max(1.0, abs(last_best_fitness)):
                stagnation_counter += 1 # Increment counter if no significant improvement
            else:
                stagnation_counter = 0 # Reset counter if improvement occurred
                last_best_fitness = best_fitness # Update last best fitness

            # If stagnation threshold is reached, diversify the population
            if stagnation_counter >= self.stagnation_threshold:
                print(f"\nStagnation detected at iteration {iteration}. Diversifying population...")
                num_to_replace = int(population_size * self.diversification_fraction)
                # Find indices of the worst solutions
                worst_indices = np.argsort(fitness_values)[-num_to_replace:]
                # Replace worst solutions with new random ones
                for idx in worst_indices:
                    initial_population[idx] = self._generate_random_solution()
                    fitness_values[idx] = self._calculate_fitness(initial_population[idx])
                # Re-evaluate the best solution after diversification
                best_idx = np.argmin(fitness_values) if np.any(np.isfinite(fitness_values)) else -1
                if best_idx != -1:
                     best_solution_Bnet = initial_population[best_idx].copy()
                     best_fitness = fitness_values[best_idx]
                # Reset stagnation tracking
                last_best_fitness = best_fitness
                stagnation_counter = 0
                print(f"Diversification complete. New best fitness: {best_fitness:.4e}")
            # --- End Stagnation Logic ---

            # Print progress periodically
            if iteration % 100 == 0 or iteration == iterations - 1:
                deviation = self._get_generator_deviation(best_solution_Bnet)
                cost = self._get_rescheduling_cost(best_solution_Bnet)
                print(f"Iter {iteration}/{iterations}: BestFit={best_fitness:.4e}, GenDev(Bnet)={deviation:.3f}, GenCost(Bnet)={cost:.2f} (Stag: {stagnation_counter}/{self.stagnation_threshold})")

        # --- End Optimization Loop ---

        # Final constraint application to the best solution found
        best_solution_Bnet = self._apply_constraints(best_solution_Bnet)

        # --- Final Reporting ---
        total_attempts_final = sum(algo_attempts_count.values())
        if total_attempts_final > 0:
            print("\nAlgorithm Contributions (Attempts):", {k: f"{v/total_attempts_final*100:.1f}%" for k, v in algo_attempts_count.items()})
        final_deviation = self._get_generator_deviation(best_solution_Bnet)
        final_gen_cost = self._get_rescheduling_cost(best_solution_Bnet)
        print(f"\nFinal Sum of Absolute Changes (Generators Only - Bnet): {final_deviation:.4f}")
        print(f"Final Generator Rescheduling Cost (Based on Bnet change): {final_gen_cost:.2f} $/hr")
        # --- --- ---

        # Calculate final flows based on the best B_net solution
        try: final_C = np.dot(self.A, best_solution_Bnet)
        except Exception: final_C = np.full((self.num_lines, 1), np.nan)

        # Return final Bnet, fitness history, and final flows
        return best_solution_Bnet, fitness_history, final_C

    # --- Memory Functions (Optional, operate on B_net) ---
    # These allow storing and potentially reusing feasible solutions from previous runs.
    def _check_memory(self, B_unused):
        """Checks memory for a previously stored feasible B_net solution."""
        if not self.memory: return None
        print("Checking memory for feasible solutions...")
        # Check recent solutions first
        for _, stored_Bnet in reversed(self.memory):
            if self._is_feasible(stored_Bnet):
                print("Using feasible B_net solution from memory.")
                return stored_Bnet
        print("No suitable feasible solution found in memory.")
        return None

    def _store_in_memory(self, initial_Bnet_state, optimized_Bnet_state):
        """Stores a feasible initial/optimized B_net pair in memory."""
        # Only store if the optimized state is feasible
        if self._is_feasible(optimized_Bnet_state):
            self.memory.append((initial_Bnet_state.copy(), optimized_Bnet_state.copy()))
            print(f"Feasible B_net solution stored. Memory size: {len(self.memory)}")
        else:
            print("Optimized solution is infeasible, not storing in memory.")

    # --- Metaheuristic Implementations (OOA, KHA, SHO - operate on B_net) ---
    # These methods implement the core logic of the different metaheuristic algorithms.
    # They take the current population (p), index of the current solution (i),
    # the best solution found so far (b), iteration number (it), and max iterations (m_it)
    # as input, and return a new candidate B_net vector.
    # They operate directly on the B_net vectors.
    def _apply_orcas_optimization(self,p,i,b,it,m_it):
        """Orcas Optimization Algorithm update step."""
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();a=2*(1-(it/m_it)**2);r1,r2=random.random(),random.random();
        if r1<0.5: # Exploitation phase
             d=np.abs(b_f-s_f);l=2*r2-1;d=np.maximum(d,1e-9);step=a*r2*(b_f-s_f);n_f=s_f+step
        else: # Exploration phase
             idx=[j for j in range(len(p)) if j!=i];X=p[random.choice(idx)].flatten() if idx else s_f;A=2*a*r1-a;C=2*r2;D=np.abs(C*X-s_f);n_f=X-A*D
        # Handle potential NaN/Inf values resulting from calculations
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2);
        return n_f.reshape(-1,1)

    def _apply_krill_herd(self,p,i,b,it,m_it):
        """Krill Herd Algorithm update step."""
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();Dmax=0.005*(1-it/m_it);Vf=.02;Nmax=.01;Dt=1.0; # KH parameters
        # Motion induced by other krill (attraction/repulsion)
        Ni=Nmax*(b_f-s_f);
        # Foraging motion (towards best food location - best solution)
        Fi=Vf*(b_f-s_f);
        # Random physical diffusion
        d=np.random.uniform(-1,1,s_f.shape);Di=Dmax*d;
        # Update position
        n_f=s_f+Dt*(Ni+Fi+Di);
        # Handle potential NaN/Inf values
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2);
        return n_f.reshape(-1,1)

    def _apply_spotted_hyena(self,p,i,b,it,m_it):
        """Spotted Hyena Optimizer update step."""
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();h=5-it*(5/m_it);B=2*random.random();E=2*h*random.random()-h; # SHO parameters
        # Encircling prey (best solution b_f)
        D_b=np.abs(B*b_f-s_f);X1=b_f-E*D_b;
        # Attacking prey (exploration/exploitation balance via E)
        if abs(E)>=1: # Exploration: move towards a random hyena
             idx=[j for j in range(len(p)) if j!=i];r_h=p[random.choice(idx)].flatten() if idx else s_f;D_h=np.abs(B*r_h-s_f);n_f=r_h-E*D_h
        else: # Exploitation: move towards the best solution
             n_f=X1
        # Handle potential NaN/Inf values
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2);
        return n_f.reshape(-1,1)


In [3]:
# Cell 3: Wrapper Function (optimize_power_flow_free_loadshed - Net Injection Model with Slack Bus)
# MODIFIED function signature: Add slack_bus_index
def optimize_power_flow_free_loadshed(A, initial_B_net, fixed_load_full,
                                      line_limits, gen_costs_full, gen_limits_min_full, gen_limits_max_full,
                                      gen_indices, load_indices,
                                      slack_bus_index, # <-- NEW required argument
                                      iterations=5000, population_size=100):
    """
    Wrapper function for Net Injection model using a SLACK BUS.
    Handles fixed loads and controllable generation.
    Objective: 1. Constraints & Min Gen Deviation (in Bnet), 2. Min Gen Cost (from Bnet change).
    Allows load shedding ONLY for buses in load_indices.

    Args:
        A (np.ndarray): System matrix (e.g., PTDF).
        initial_B_net (np.ndarray): Initial NET bus injections (Pg - Pl).
        fixed_load_full (np.ndarray): Fixed load demand (Pl >= 0) for ALL buses.
        line_limits (np.ndarray): Absolute limits for line flows.
        gen_costs_full (np.ndarray): Cost coefficients for ALL buses.
        gen_limits_min_full (np.ndarray): Min generation (Pg) limit for ALL buses.
        gen_limits_max_full (np.ndarray): Max generation (Pg) limit for ALL buses.
        gen_indices (list or np.ndarray): 0-based indices for generator buses.
        load_indices (list or np.ndarray): 0-based indices for load-only buses.
        slack_bus_index (int): 0-based index of the designated slack bus (must be in gen_indices).
        iterations (int): Number of optimization iterations.
        population_size (int): Number of solutions in the population.

    Returns:
        tuple: (B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details)
               - B_optimized_net: Optimized NET bus injections.
               - fitness_history: List of best fitness values per iteration.
               - C_optimized: Optimized line flows corresponding to B_optimized_net.
               - C_unoptimized: Line flows corresponding to the initial (potentially adjusted) B_net.
               - final_feasible: Boolean indicating if the final solution is feasible.
               - opt_details: Dictionary containing optimizer setup details (indices, load, slack index).
    """
    print("--- Initializing Optimizer (Net Injection Model, SLACK BUS Balancing) ---")
    try:
        # Check if the optimizer class is defined
        if 'HybridPowerFlowOptimizer' not in globals():
            raise NameError("Optimizer class 'HybridPowerFlowOptimizer' is not defined.")

        # Instantiate the optimizer, passing the slack_bus_index
        optimizer = HybridPowerFlowOptimizer(A, line_limits, gen_costs_full, gen_limits_min_full, gen_limits_max_full,
                                             initial_B_net, fixed_load_full,
                                             gen_indices, load_indices,
                                             slack_bus_index) # <-- Pass slack index

    except (ValueError, NameError, IndexError) as e:
        # Handle errors during initialization (e.g., invalid slack index, index mismatch)
        print(f"ERROR initializing optimizer: {e}")
        # Calculate unoptimized flows based on the raw initial B_net for reporting
        try: C_unoptimized_calc = np.dot(A, initial_B_net)
        except Exception: C_unoptimized_calc = np.full((A.shape[0], 1), np.nan)
        # Return initial state and indicate failure
        return initial_B_net, [], C_unoptimized_calc, C_unoptimized_calc, False, {}

    # Get the potentially adjusted initial B_net from the optimizer instance
    initial_Bnet_from_opt = optimizer.initial_B_net.reshape(-1, 1)
    # Calculate unoptimized flows based on this adjusted initial state
    try: C_unoptimized = np.dot(A, initial_Bnet_from_opt)
    except Exception as e:
        print(f"Warning: Could not calculate unoptimized flows: {e}")
        C_unoptimized = np.full((A.shape[0], 1), np.nan)

    print("\n--- Checking Initial State Feasibility (Using Optimizer's Initial B_net) ---")
    # Check if the state the optimizer starts with is feasible
    initial_feasible = optimizer._is_feasible(initial_Bnet_from_opt, verbose=True)
    print(f"Optimizer's initial state feasible: {initial_feasible}")
    if not initial_feasible:
        print("WARNING: Optimizer starting from an infeasible state (after initial adjustments).")

    print("\n--- Starting Optimization (Net Injection Model with Slack Bus) ---")
    # Run the optimization process
    # The first argument to optimize (B_unused) is ignored in this implementation
    B_opt_net, fit_hist, C_opt = optimizer.optimize(None, iterations=iterations, population_size=population_size)

    print("\n--- Checking Final Solution Feasibility ---")
    # Check if the final optimized solution meets all constraints
    final_feasible = optimizer._is_feasible(B_opt_net, verbose=True)
    print(f"\nFinal feasibility: {final_feasible}")

    # Store relevant details from the optimizer setup
    details = {
        "gen_indices": optimizer.gen_indices,
        "load_indices": optimizer.load_indices,
        "fixed_load": optimizer.fixed_load,
        "slack_bus_index": optimizer.slack_bus_index # Include slack index in details
    }

    # Return the optimized results
    return B_opt_net, fit_hist, C_opt, C_unoptimized, final_feasible, details


In [4]:
# Cell 4: Visualization Function (visualize_results - Minor changes for robustness)
def visualize_results(A, B, B_optimized, C_optimized, C_unoptimized, line_limits,
                      gen_costs_full, gen_limits_min_full, gen_limits_max_full, # Use full arrays for limits display
                      gen_indices, load_indices, slack_bus_index, # Add slack index for context
                      fitness_history):
    """
    Visualizes the optimization results (Net Injection Model with Slack Bus).
    Includes checks for valid data before plotting.

    Args:
        A (np.ndarray): System matrix.
        B (np.ndarray): Initial net injection vector (potentially adjusted).
        B_optimized (np.ndarray): Optimized net injection vector.
        C_optimized (np.ndarray): Optimized line flows.
        C_unoptimized (np.ndarray): Initial line flows.
        line_limits (np.ndarray): Line power limits.
        gen_costs_full (np.ndarray): Full array of generator costs (used for summary).
        gen_limits_min_full (np.ndarray): Full array of min Pg limits (used for plotting).
        gen_limits_max_full (np.ndarray): Full array of max Pg limits (used for plotting).
        gen_indices (np.ndarray): Indices of generator buses.
        load_indices (np.ndarray): Indices of load-only buses.
        slack_bus_index (int): Index of the slack bus.
        fitness_history (list): History of best fitness values.
    """
    num_lines = A.shape[0]
    num_buses = A.shape[1]
    tolerance = 1e-6 # Tolerance for violation checks

    print("\n--- Generating Plots (Net Injection Model - Slack Bus Balancing) ---")

    # Plot 1: Line Flows Comparison
    try:
        plt.figure(figsize=(12, 6))
        idx = np.arange(1, num_lines + 1) # Line indices for plotting (1-based)

        # Plot unoptimized flows if valid
        if isinstance(C_unoptimized, np.ndarray) and C_unoptimized.size == num_lines and np.all(np.isfinite(C_unoptimized)):
            plt.plot(idx, C_unoptimized.flatten(), 'o-', label='Initial Flows', alpha=0.7, markersize=4)
        else: print("Warning: Skipping initial flows plot (invalid data).")

        # Plot optimized flows if valid
        if isinstance(C_optimized, np.ndarray) and C_optimized.size == num_lines and np.all(np.isfinite(C_optimized)):
            plt.plot(idx, C_optimized.flatten(), 's--', label='Optimized Flows', alpha=0.9, markersize=5)
            # Check for violations in optimized flows
            flows_opt_abs = np.abs(C_optimized.flatten())
            violations = np.where(flows_opt_abs > line_limits + tolerance)[0]
            if len(violations) > 0:
                # Highlight violations on the plot
                plt.scatter(idx[violations], C_optimized.flatten()[violations], c='magenta', s=100, zorder=5, label=f'Violations ({len(violations)})', marker='x')
        else: print("Warning: Skipping optimized flows plot (invalid data).")

        # Plot line limits
        plt.plot(idx, line_limits, 'r:', alpha=0.8, label='Limit (+)')
        plt.plot(idx, -line_limits, 'r:', alpha=0.8, label='Limit (-)')

        plt.xlabel('Line Index')
        plt.ylabel('Power Flow (MW or p.u.)')
        plt.title(f'Line Flows Comparison (Slack Bus: {slack_bus_index+1})')
        plt.legend()
        plt.grid(True, linestyle=':')
        plt.xticks(idx[::max(1, num_lines//20)]) # Adjust x-ticks density for readability
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plotting error (Line Flows): {e}")

    # Plot 2: Fitness History (Convergence Plot)
    try:
        # Check if fitness_history is a list/array with more than one point
        if isinstance(fitness_history, (list, np.ndarray)) and len(fitness_history) > 1:
            plt.figure(figsize=(10, 5))
            # Filter out non-finite values for plotting robustness
            finite_fitness = [f for f in fitness_history if np.isfinite(f)]
            if len(finite_fitness) > 1:
                plt.plot(finite_fitness, '.-', color='royalblue', label='Best Fitness', markersize=3, linewidth=1)
                plt.xlabel('Iteration')
                plt.ylabel('Fitness Value (Log Scale)')
                plt.title('Optimization Convergence')
                plt.yscale('log') # Use log scale for potentially large fitness values/penalties
                plt.legend()
                plt.grid(True, linestyle=':')
                plt.tight_layout()
                plt.show()
            else: print("Warning: Not enough finite fitness values to plot convergence.")
        else: print("Warning: Skipping fitness plot (insufficient history data).")
    except Exception as e:
        print(f"Plotting error (Fitness History): {e}")

    # Plot 3: Bus Net Injections (B_net) Comparison
    try:
        plt.figure(figsize=(14, 7))
        bus_idx_plot = np.arange(1, num_buses + 1) # Bus indices for plotting (1-based)
        bar_width = 0.35

        # Define colors based on generator/load status, highlight slack bus
        colors_initial = []
        colors_optimized = []
        for i in range(num_buses):
             if i == slack_bus_index:
                 colors_initial.append('darkred')
                 colors_optimized.append('red')
             elif i in gen_indices:
                 colors_initial.append('darkblue')
                 colors_optimized.append('darkgreen')
             else: # Load bus
                 colors_initial.append('skyblue')
                 colors_optimized.append('lightgreen')


        # Plot initial and optimized B_net injections
        valid_B = isinstance(B, np.ndarray) and B.size == num_buses and np.all(np.isfinite(B))
        valid_B_opt = isinstance(B_optimized, np.ndarray) and B_optimized.size == num_buses and np.all(np.isfinite(B_optimized))

        if valid_B:
            plt.bar(bus_idx_plot - bar_width/2, B.flatten(), width=bar_width, label='Initial B_net (Slack=Red)', alpha=0.7, color=colors_initial)
        else: print("Warning: Skipping initial B_net plot (invalid data).")

        if valid_B_opt:
            plt.bar(bus_idx_plot + bar_width/2, B_optimized.flatten(), width=bar_width, label='Optimized B_net (Slack=Red)', alpha=0.8, color=colors_optimized)
        else: print("Warning: Skipping optimized B_net plot (invalid data).")

        # --- Plot Generator Pg Limits (Optional, requires fixed load data) ---
        # Note: Plotting Pg limits directly on B_net plot can be confusing.
        # Consider a separate plot for Pg if needed. We'll skip adding Bnet limits here
        # as they depend on fixed load and make the plot cluttered.

        plt.xlabel('Bus Index')
        plt.ylabel('Net Power Injection (B_net = Pg - Pl)')
        plt.title(f'Bus Net Power Injections: Initial vs. Optimized (Slack Bus: {slack_bus_index+1})')
        plt.xticks(bus_idx_plot[::max(1, num_buses//20)]) # Adjust x-ticks density
        plt.legend()
        plt.grid(True, axis='y', linestyle=':')
        plt.axhline(0, color='black', linewidth=0.5) # Zero line for reference
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plotting error (Bus Injections): {e}")

    # Plot 4 & 5: Changes in Generation and Load Shedding
    try:
        # Ensure B and B_optimized are valid before calculating changes
        if valid_B and valid_B_opt:
            changes = B_optimized.flatten() - B.flatten() # Change in B_net
            bus_idx_plot = np.arange(1, num_buses + 1)

            # --- Generator Changes Plot (Change in B_net for Generators) ---
            if len(gen_indices) > 0:
                plt.figure(figsize=(10, 4))
                gen_changes = changes[gen_indices]
                gen_labels = bus_idx_plot[gen_indices]
                # Color bars based on increase/decrease, highlight slack
                colors_gen_change = []
                for idx in gen_indices:
                    if idx == slack_bus_index:
                        colors_gen_change.append('red' if gen_changes[np.where(gen_indices==idx)[0][0]] >= 0 else 'darkred')
                    else:
                        colors_gen_change.append('forestgreen' if gen_changes[np.where(gen_indices==idx)[0][0]] >= 0 else 'firebrick')

                bar_indices_gen = np.arange(len(gen_indices))
                plt.bar(bar_indices_gen, gen_changes, color=colors_gen_change)
                plt.axhline(0, color='black', linestyle='-', linewidth=0.7)
                plt.xlabel('Generator Bus Index')
                plt.ylabel('Change in Net Injection (ΔB_net)')
                plt.title(f'Generator Net Injection Changes (Optimized - Initial) (Slack: {slack_bus_index+1})')
                plt.xticks(bar_indices_gen, gen_labels)
                plt.grid(True, axis='y', linestyle=':')
                plt.tight_layout()
                plt.show()

            # --- Load Changes (Shedding) Plot ---
            if len(load_indices) > 0:
                plt.figure(figsize=(10, 4))
                # Positive change for load means injection increased (less negative) -> load shed
                load_changes = changes[load_indices]
                # Load Shed = max(0, Final_Bnet - Initial_Bnet) for load buses
                # This represents the amount the load was reduced (Bnet moved towards 0)
                load_shed = np.maximum(0, load_changes)
                load_labels = bus_idx_plot[load_indices]
                # Color bars where shedding occurred
                colors_load = ['darkorange' if x > tolerance else 'darkgrey' for x in load_shed]
                bar_indices_load = np.arange(len(load_indices))
                plt.bar(bar_indices_load, load_shed, color=colors_load)
                plt.axhline(0, color='black', linestyle='-', linewidth=0.7)
                plt.xlabel('Load Bus Index')
                plt.ylabel('Load Shed Amount (MW or p.u.)')
                plt.title('Load Shedding (Optimized B_net - Initial B_net)')
                plt.xticks(bar_indices_load, load_labels)
                plt.grid(True, axis='y', linestyle=':')
                # Adjust y-axis slightly below zero for visibility
                plt.ylim(bottom= -0.05 * max(1, np.max(load_shed)) if np.any(load_shed > 0) else -0.1)
                plt.tight_layout()
                plt.show()
        else:
            print("Warning: Skipping change/shedding plots (invalid B or B_optimized data).")

    except Exception as e:
        print(f"Plotting error (Changes/Shedding): {e}")

    # --- Text Summary ---
    # (This part requires re-instantiating the optimizer or passing it, which is complex.
    #  We will calculate metrics directly here based on inputs.)
    print("\n" + "="*30 + " RESULTS SUMMARY (Slack Bus Model) " + "="*30)
    np.set_printoptions(precision=4, suppress=True) # Adjust numpy print precision
    try:
        # Calculate final metrics only if data is valid
        final_dev_g = np.nan
        final_g_cost = np.nan
        total_ls = np.nan

        if valid_B and valid_B_opt and isinstance(gen_costs_full, np.ndarray) and gen_costs_full.size == num_buses:
            if len(gen_indices) > 0:
                g_costs_only = gen_costs_full.flatten()[gen_indices]
                # Calculate deviation and cost based on B_net changes for ALL generators
                gen_deviation_vector = np.abs(B_optimized.flatten()[gen_indices] - B.flatten()[gen_indices])
                final_dev_g = np.sum(gen_deviation_vector)
                final_g_cost = np.sum(g_costs_only * gen_deviation_vector)

            if len(load_indices) > 0:
                load_shed_amount = np.maximum(0, B_optimized.flatten()[load_indices] - B.flatten()[load_indices])
                total_ls = np.sum(load_shed_amount)
            else:
                 total_ls = 0.0 # No load buses means no load shed

        print(f"\nInitial B_net (Used by Optimizer):\n{B.flatten() if valid_B else 'N/A'}")
        print(f"\nOptimized B_net:\n{B_optimized.flatten() if valid_B_opt else 'N/A'}")
        if valid_B_opt: print(f"Sum Optimized B_net: {np.sum(B_optimized):.6f}") # Should be very close to 0

        print(f"\nInitial Flows (C_unopt):\n{C_unoptimized.flatten() if isinstance(C_unoptimized, np.ndarray) and C_unoptimized.size == num_lines else 'N/A'}")
        print(f"\nOptimized Flows (C_opt):\n{C_optimized.flatten() if isinstance(C_optimized, np.ndarray) and C_optimized.size == num_lines else 'N/A'}")
        print("-" * 70)
        print("\nObjective Metrics & Load Shed:")
        print(f"  Generator Deviation Sum (All Gens, B_net based): {final_dev_g:.4f}")
        print(f"  Generator Rescheduling Cost (All Gens, B_net based): {final_g_cost:.2f}")
        print(f"  Total Load Shed (at Load-Only buses): {total_ls:.4f} MW (or p.u.)")

        print("\nConstraint Check Summary (from Optimizer):")
        # Feasibility is checked within the wrapper function after optimization.
        # Here we just report the result passed to this function.
        # Re-checking feasibility here would require passing the optimizer object or all its parameters.
        # We rely on the feasibility check done immediately after the optimize() call.
        # (Note: The feasibility check within visualize_results in the original code was removed
        #  as it required re-instantiating the optimizer unnecessarily).
        # We need to get the feasibility status from the wrapper function's return value.
        # Let's assume 'final_feasible' boolean is passed correctly.
        # final_feasible = ... # This value should be passed from the main script execution
        # print(f"  Final solution feasible (reported by optimizer): {'YES' if final_feasible else 'NO'}")


    except Exception as e:
        print(f"Error generating summary text: {e}")

    finally:
        np.set_printoptions(precision=8, suppress=False) # Reset numpy print options

    print("=" * 70)
    print("Note: Visualization assumes MW or consistent p.u. units. Voltage constraints not included.")
    print(f"Slack Bus used for balancing: {slack_bus_index+1}")
    print("=" * 70)


In [5]:
# Cell 5: Main Execution Block (Net Injection Model with Slack Bus)
if __name__ == "__main__":

    # --- Ensure functions/classes from previous cells are available ---
    # In a real notebook, ensure Cells 1-4 have been executed
    # In a script, they are defined above. Basic check:
    if 'HybridPowerFlowOptimizer' not in globals(): print("FATAL ERROR: HybridPowerFlowOptimizer class not defined."); exit()
    if 'optimize_power_flow_free_loadshed' not in globals(): print("FATAL ERROR: optimize_power_flow_free_loadshed function not defined."); exit()
    if 'visualize_results' not in globals(): print("FATAL ERROR: visualize_results function not defined."); exit()


    # --- Load System Matrix A ---
    matrix_file_name = 'reshaped_data.csv' # Make sure this file exists in the same directory
    try:
        print(f"Loading system matrix A from: {matrix_file_name}")
        df = pd.read_csv(matrix_file_name, header=None)
        A = df.to_numpy()
        print(f"Successfully loaded matrix A with shape {A.shape}")
        if A.ndim != 2 or A.shape[0] == 0 or A.shape[1] == 0:
             raise ValueError("Invalid matrix A shape or dimensions.")
    except FileNotFoundError:
        print(f"FATAL ERROR: Matrix file '{matrix_file_name}' not found. Please create it or specify the correct path.")
        exit()
    except Exception as e:
        print(f"FATAL ERROR loading matrix A: {e}. Exiting.")
        exit()

    num_lines_main = A.shape[0]
    num_buses_main = A.shape[1]
    print(f"System dimensions: {num_lines_main} lines, {num_buses_main} buses.")
    all_bus_indices_set = set(range(num_buses_main)) # Set of all 0-based bus indices

    # --- Main Scenario Loop ---
    while True:
        print("\n" + "="*25 + " New Scenario (Net Injection - Slack Bus) " + "="*25)
        print("Objective: 1. Meet Limits & Min Gen Deviation, 2. Min Gen Cost")
        print("(Load Shedding Allowed for Loads, SLACK BUS Balancing)") # Update description

        try:
            # --- Get User Inputs ---

            # 1. Get Generator Indices (1-based)
            temp_gen_indices_0based = [] # List to store 0-based generator indices
            while True:
                try:
                    gen_indices_str = input(f"\n>>> Enter GENERATOR bus numbers (1 to {num_buses_main}, space-separated): ")
                    user_gen_indices_list = [int(x) for x in gen_indices_str.split()]
                    valid_indices = True
                    gen_indices_set = set() # To check for duplicates
                    temp_gen_indices_0based_current = []
                    for idx_1based in user_gen_indices_list:
                        if 1 <= idx_1based <= num_buses_main:
                            idx_0based = idx_1based - 1
                            if idx_0based in gen_indices_set:
                                print(f"  Error: Duplicate index {idx_1based}.")
                                valid_indices = False; break
                            gen_indices_set.add(idx_0based)
                            temp_gen_indices_0based_current.append(idx_0based)
                        else:
                            print(f"  Error: Invalid bus number {idx_1based}. Must be between 1 and {num_buses_main}.")
                            valid_indices = False; break
                    if valid_indices:
                        temp_gen_indices_0based = sorted(temp_gen_indices_0based_current)
                        break # Exit loop if input is valid
                except ValueError:
                    print("  Error: Invalid input format. Please enter space-separated numbers.")
                except EOFError: raise # Allow breaking with Ctrl+D/Ctrl+Z

            # Handle case of no generators immediately - need at least one for slack bus
            if not temp_gen_indices_0based:
                 print("ERROR: At least one generator must be specified to assign a slack bus. Restarting...")
                 continue # Restart scenario loop

            # --- NEW: Get Slack Bus Index (1-based) ---
            slack_bus_index_input = -1 # Initialize with invalid value
            while True:
                try:
                    slack_bus_1based_str = input(f">>> Enter SLACK BUS number (must be one of {np.array(temp_gen_indices_0based) + 1}): ")
                    slack_bus_1based = int(slack_bus_1based_str)
                    slack_bus_index_input_candidate = slack_bus_1based - 1
                    # Check if the entered slack bus is in the list of generators
                    if slack_bus_index_input_candidate in temp_gen_indices_0based:
                        slack_bus_index_input = slack_bus_index_input_candidate
                        print(f"-> Using Bus {slack_bus_1based} as Slack Bus.")
                        break # Exit loop if valid slack bus entered
                    else:
                        print(f"  Error: Slack bus {slack_bus_1based} is not in the list of specified generators {np.array(temp_gen_indices_0based) + 1}.")
                except ValueError:
                    print("  Error: Invalid number format. Please enter a single number.")
                except EOFError:
                    raise # Allow breaking


            # Determine load-only indices (all buses not designated as generators)
            temp_load_indices_0based = sorted(list(all_bus_indices_set - set(temp_gen_indices_0based)))
            print(f"-> Using Generators: {np.array(temp_gen_indices_0based) + 1}")
            print(f"-> Using Load-Only Buses: {np.array(temp_load_indices_0based) + 1}")

            # 2. Get Fixed Load (Pl) for ALL buses
            print(f"\n>>> Enter FIXED LOAD demand (Pl >= 0) for ALL buses ({num_buses_main} values, space-separated):")
            print("    (Enter 0 for buses with no load)")
            fixed_load_input = np.zeros(num_buses_main)
            while True:
                try:
                    pl_str = input("  Fixed Loads (Pl): ")
                    pl_list = [float(x) for x in pl_str.split()]
                    if len(pl_list) == num_buses_main:
                        fixed_load_input = np.array(pl_list)
                        if np.any(fixed_load_input < 0):
                            print("  Warning: Negative loads entered. Treating them as zero load.")
                            fixed_load_input = np.maximum(0, fixed_load_input) # Ensure Pl >= 0
                        break # Exit loop if valid input
                    else:
                        print(f"  Error: Expected {num_buses_main} values, got {len(pl_list)}.")
                except ValueError:
                    print("  Error: Invalid number format. Please enter space-separated numbers.")
                except EOFError: raise

            # 3. Get Initial Generation (Pg_initial) ONLY for GENERATOR buses
            pg_initial_input = np.zeros(num_buses_main) # Initialize Pg for all buses to 0
            if len(temp_gen_indices_0based) > 0:
                print(f"\n>>> Enter INITIAL GENERATION (Pg) ONLY for GENERATOR buses {np.array(temp_gen_indices_0based) + 1}:")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                            pg_val_str = input(f"    G{i+1} Initial Pg: ")
                            pg_initial_input[i] = float(pg_val_str)
                            break # Exit inner loop for this generator
                        except ValueError:
                            print("    Invalid number.")
                        except EOFError:
                            raise
            else: print("\nNOTE: No generators specified (this shouldn't happen due to earlier check).")

            # 4. Calculate Initial NET Injection (B = Pg - Pl)
            B_input_net = (pg_initial_input - fixed_load_input).reshape(-1, 1)
            print("\nCalculated Initial Net Injections (B = Pg - Pl):")
            for i in range(num_buses_main): print(f"  Bus {i+1}: {B_input_net[i,0]:.4f}")

            # 5. Get Line Limits
            print(f"\n>>> Enter LINE power limits ({num_lines_main} values, space-separated):")
            line_limits_input = np.zeros(num_lines_main)
            while True:
                try:
                    limits_str = input("  Line limits: ")
                    line_limits_list = [float(x) for x in limits_str.split()]
                    if len(line_limits_list) == num_lines_main:
                        line_limits_input = np.abs(np.array(line_limits_list)) # Use absolute value for limits
                        break # Exit loop
                    else:
                        print(f"  Error: Expected {num_lines_main} values, got {len(line_limits_list)}.")
                except ValueError:
                    print("  Error: Invalid number format.")
                except EOFError: raise

            # 6. Get Generator-Specific Data (Costs, Pg Limits)
            gen_costs_full_input = np.zeros(num_buses_main)
            gen_limits_pg_min_input = np.zeros(num_buses_main)
            gen_limits_pg_max_input = np.zeros(num_buses_main)
            if len(temp_gen_indices_0based) > 0:
                print(f"\n>>> Enter Cost/Limits ONLY for GENERATOR buses {np.array(temp_gen_indices_0based) + 1}:")
                print("  Enter Gen Cost Coefficients ($/MWh change):")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                             cost_str = input(f"    G{i+1} cost: ")
                             gen_costs_full_input[i] = float(cost_str)
                             break
                        except ValueError: print("    Invalid number.")
                        except EOFError: raise
                print("  Enter Gen MIN Generation Limit (Pg_min MW):")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                            min_pg_str = input(f"    G{i+1} Pg_min: ")
                            gen_limits_pg_min_input[i] = float(min_pg_str)
                            break
                        except ValueError: print("    Invalid number.")
                        except EOFError: raise
                print("  Enter Gen MAX Generation Limit (Pg_max MW):")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                            max_pg_str = input(f"    G{i+1} Pg_max: ")
                            max_val = float(max_pg_str)
                            # Validate max >= min
                            if max_val < gen_limits_pg_min_input[i]:
                                print(f"    Error: Max Pg ({max_val}) < Min Pg ({gen_limits_pg_min_input[i]}). Re-enter Max.")
                                continue # Ask again for max limit
                            gen_limits_pg_max_input[i] = max_val
                            break # Exit inner loop
                        except ValueError: print("    Invalid number.")
                        except EOFError: raise
            # No else needed as we ensured generators exist

        except EOFError: print("\nInput interrupted during setup. Restarting scenario..."); continue
        except Exception as e: print(f"\nAn unexpected error occurred during input: {e}. Restarting..."); continue

        # --- Call Optimization ---
        print("\n" + "="*25 + " Running Optimization (SLACK BUS Model) " + "="*25)
        B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details = (None, [], None, None, False, {}) # Default values
        try:
            opt_iterations = 5000 # Number of iterations for the optimizer
            opt_pop_size = 250   # Population size for the metaheuristics
            start_time = time.time()

            # Call the wrapper function, passing the selected slack bus index
            B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details = optimize_power_flow_free_loadshed(
                A, B_input_net, fixed_load_input,
                line_limits_input, gen_costs_full_input, gen_limits_pg_min_input, gen_limits_pg_max_input,
                temp_gen_indices_0based, temp_load_indices_0based,
                slack_bus_index_input, # <-- Pass the validated slack bus index
                iterations=opt_iterations,
                population_size=opt_pop_size
            )
            end_time = time.time()
            print(f"Optimization Duration: {end_time - start_time:.2f} seconds")

        except NameError as e:
            print(f"FATAL ERROR: Required function/class not defined ({e}). Exiting script."); break
        except Exception as e:
            print(f"An unexpected error occurred during optimization run: {e}")
            # Ask user if they want to continue despite the error
            try:
                if input("Try another scenario anyway? (y/n):").strip().lower() != 'y':
                    break # Exit main loop
                else:
                    continue # Go to next iteration of main loop
            except EOFError: print("\nInput interrupted. Exiting..."); break

        print("\n" + "="*25 + " Optimization Finished " + "="*25)

        # --- Post-processing: Calculate Optimized Pg and Load Shed from B_net ---
        # Use details returned from the optimizer function
        final_fixed_load = opt_details.get("fixed_load", np.zeros(num_buses_main))
        final_gen_indices = opt_details.get("gen_indices", np.array([]))
        final_load_indices = opt_details.get("load_indices", np.array([]))
        final_slack_index = opt_details.get("slack_bus_index", -1)

        optimized_Pg = np.zeros(num_buses_main)
        load_shed_vector = np.zeros(num_buses_main)

        if B_optimized_net is not None and final_feasible: # Only calculate if optimization succeeded and was feasible
            B_opt_flat = B_optimized_net.flatten()
            # Calculate Pg for generators: Pg = B_net + Pl
            if len(final_gen_indices) > 0:
                 optimized_Pg[final_gen_indices] = B_opt_flat[final_gen_indices] + final_fixed_load[final_gen_indices]

            # Calculate Load Shed for load buses: Shed = Pl_initial - Pl_final = Pl_initial - (Pg - B_net)
            # Since Pg=0 for load buses, Pl_final = -B_net
            # Shed = Pl_initial - (-B_net_final) = Pl_initial + B_net_final
            # Note: B_net_final is <= 0 for load buses. Pl_initial is fixed_load.
            if len(final_load_indices) > 0:
                 load_shed_vector[final_load_indices] = final_fixed_load[final_load_indices] + B_opt_flat[final_load_indices]
                 # Ensure shed amount is non-negative (due to potential tolerance issues)
                 load_shed_vector = np.maximum(0, load_shed_vector)

        # --- Visualize Results ---
        try:
            print("\nVisualizing results...")
            # Pass necessary data to the visualization function
            visualize_results(A,
                              B_input_net, # Use the initial B_net provided to optimizer
                              B_optimized_net,
                              C_optimized,
                              C_unoptimized, # Flows corresponding to initial B_net
                              line_limits_input,
                              gen_costs_full_input,
                              gen_limits_pg_min_input, # Pass full arrays for plotting context
                              gen_limits_pg_max_input,
                              final_gen_indices, final_load_indices, final_slack_index, # Pass final indices
                              fitness_history)
        except NameError as e:
            print(f"Error during visualization: Required function not defined ({e}).")
        except Exception as e:
            print(f"An error occurred during visualization: {e}")

        # --- Save Results ---
        if final_feasible and B_optimized_net is not None: # Only save if feasible and optimization ran
            try:
                save = input("\nSave detailed results to file? (y/n): ").strip().lower()
                if save == 'y':
                    default_fname = "power_flow_slack_bus_results.txt"
                    fname = input(f"Enter filename (default: {default_fname}): ").strip() or default_fname
                    print(f"Attempting to save results to {fname}...")
                    try:
                        with open(fname, 'w') as f:
                            # Get details needed for summary (use final indices from opt_details)
                            save_gen_indices = final_gen_indices
                            save_load_indices = final_load_indices
                            save_slack_idx = final_slack_index
                            gen_dev = np.nan; gen_cost = np.nan; total_load_shed = np.sum(load_shed_vector)
                            gen_costs_save = []

                            # Recalculate metrics based on final B_net
                            if len(save_gen_indices) > 0:
                                gen_costs_save=gen_costs_full_input[save_gen_indices]
                                dev_vec = np.abs(B_optimized_net.flatten()[save_gen_indices] - B_input_net.flatten()[save_gen_indices])
                                gen_dev = np.sum(dev_vec)
                                gen_cost = np.sum(gen_costs_save * dev_vec)

                            # Write Header
                            f.write("Power Flow Opt Results (Net Injection Model, SLACK BUS Balancing)\n")
                            f.write("Objective: 1. Limits & Min Gen Deviation, 2. Min Gen Cost\n")
                            f.write(f"SLACK BUS: Bus {save_slack_idx+1}\n" if save_slack_idx != -1 else "SLACK BUS: None\n")
                            f.write("="*30+"\n\n")

                            # Write Inputs
                            f.write(f"Matrix A (Shape: {A.shape}):\n"); np.savetxt(f, A, fmt='%.4f'); f.write("\n")
                            f.write("Line Limits:\n"); [f.write(f"L{i+1}: {l:.2f}\n") for i,l in enumerate(line_limits_input)]
                            f.write("\nFixed Loads (Pl):\n"); [f.write(f"Bus {i+1}: {pl:.4f}\n") for i,pl in enumerate(final_fixed_load)]
                            f.write("\nGen Costs ($/MW change):\n"); [f.write(f"G{save_gen_indices[i]+1}: {c:.2f}\n") for i,c in enumerate(gen_costs_save)]
                            f.write("\nGen Min Generation (Pg_min):\n"); [f.write(f"G{idx+1}: {gen_limits_pg_min_input[idx]:.2f}\n") for idx in save_gen_indices]
                            f.write("\nGen Max Generation (Pg_max):\n"); [f.write(f"G{idx+1}: {gen_limits_pg_max_input[idx]:.2f}\n") for idx in save_gen_indices]

                            # Write Initial and Optimized States
                            f.write("\nInitial Net Injection (B_net = Pg_initial - Pl):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_input_net)]
                            f.write("\nOptimized Net Injection (B_net):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_optimized_net)]
                            f.write("\nOptimized Generation (Pg = B_net_opt + Pl at Gen buses):\n"); [f.write(f"Bus {idx+1}: {optimized_Pg[idx]:.4f}\n") for idx in save_gen_indices]
                            f.write("\nLoad Shed Amount (at Load-Only buses):\n"); [f.write(f"Bus {idx+1}: {load_shed_vector[idx]:.4f}\n") for idx in save_load_indices if load_shed_vector[idx] > tolerance] # Only show where shed > 0
                            f.write("\nInitial Flows (C_unopt = A * B_net_initial):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_unoptimized)]
                            f.write("\nOptimized Flows (C_opt = A * B_net_opt):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_optimized)]

                            # Write Summary Metrics
                            f.write(f"\nFinal Gen Deviation Sum (B_net based): {gen_dev:.4f}\n")
                            f.write(f"Final Gen Rescheduling Cost (B_net based): {gen_cost:.2f}\n")
                            f.write(f"Total Load Shed (at Load-Only buses): {total_load_shed:.4f} MW (or p.u.)\n")
                            f.write(f"\nFeasible: {'YES' if final_feasible else 'NO'}\n")

                        print(f"Results successfully saved to {fname}")
                    except IOError as e: print(f"ERROR saving results to file '{fname}': {e}")
                    except IndexError as e: print(f"ERROR saving results: Index out of bounds - {e}.")
                    except Exception as e: print(f"An unexpected error occurred during saving: {e}")
            except EOFError:
                print("\nInput interrupted during save prompt.")
                # Ask again if user wants to continue
                try:
                     if input("Continue to next scenario anyway? (y/n):").strip().lower() != 'y': break
                except EOFError: break # Exit if interrupted again
        elif not final_feasible:
             print("\nFinal solution was infeasible. Results not saved.")
        else: # Case where optimization didn't run or returned None
             print("\nOptimization did not produce a valid result. Nothing to save.")


        # --- Ask to run again ---
        try:
            run_again = input("\nRun another scenario? (y/n):").strip().lower()
            if run_again != 'y':
                print("\nExiting...")
                break # Exit the main while loop
            else:
                print("\nRestarting scenario...\n" + "-"*70)
        except EOFError:
            print("\nInput interrupted. Exiting...")
            break # Exit the main while loop

    # --- End of Script ---
    print("\nScript finished.")


Loading system matrix A from: reshaped_data.csv
Successfully loaded matrix A with shape (41, 30)
System dimensions: 41 lines, 30 buses.

========================= New Scenario (Net Injection - Slack Bus) =========================
Objective: 1. Meet Limits & Min Gen Deviation, 2. Min Gen Cost
(Load Shedding Allowed for Loads, SLACK BUS Balancing)


KeyboardInterrupt: Interrupted by user